In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# options d'affichage
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
sns.set_theme(style="whitegrid")

In [ ]:
df_train = pd.read_csv('../data/application_train.csv')

display(df_train.shape)
df_train.head()

In [ ]:
# distribution de la variable cible
print("=== Distribution de la variable cible ===")
target_counts = df_train['TARGET'].value_counts()
target_pct = df_train['TARGET'].value_counts(normalize=True) * 100

print(f"Classe 0 (bon client)  : {target_counts[0]:>6} ({target_pct[0]:.1f}%)")
print(f"Classe 1 (défaut)      : {target_counts[1]:>6} ({target_pct[1]:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 5))  # un peu plus haut
ax.bar(['Bon client (0)', 'Défaut (1)'], target_counts, color=['steelblue', 'salmon'])
ax.set_title("Distribution de la variable cible", pad=20)  # pad éloigne le titre
ax.set_ylabel("Nombre de clients")
ax.set_ylim(0, target_counts[0] * 1.15)  # on agrandit l'axe Y pour laisser de la place
for i, v in enumerate(target_counts):
    ax.text(i, v + 1000, f"{v}\n({target_pct[i]:.1f}%)", ha='center')
plt.tight_layout()
plt.show()

In [ ]:
print("=== Types de variables ===")
print(df_train.dtypes.value_counts())

print(f"\n=== Valeurs manquantes ===")
missing = df_train.isnull().sum()
missing_pct = (missing / len(df_train)) * 100
missing_df = pd.DataFrame({
    'nb_manquants': missing,
    'pct_manquants': missing_pct
}).sort_values('pct_manquants', ascending=False)

print(missing_df[missing_df['nb_manquants'] > 0].head(20))

In [ ]:
# analyse des corrélations avec la TARGET

# on sélectionne uniquement les colonnes numériques
num_cols_corr = df_train.select_dtypes(include=['float64', 'int64']).columns.tolist()

# calcul des corrélations avec la TARGET
correlations = df_train[num_cols_corr].corr()['TARGET'].drop('TARGET')
correlations = correlations.abs().sort_values(ascending=False)

print("=== Top 15 features les plus corrélées avec TARGET ===")
print(correlations.head(15).round(3))

# visualisation
fig, ax = plt.subplots(figsize=(10, 6))
correlations.head(15).plot(kind='bar', ax=ax, color='steelblue')
ax.set_title("Top 15 - Corrélations (valeur absolue) avec TARGET")
ax.set_xlabel("")
ax.set_ylabel("Corrélation absolue")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# matrice de corrélation entre features (heatmap)

# on prend uniquement les features les plus importantes
# pour garder un graphique lisible
top_features = correlations.head(15).index.tolist() + ['TARGET']

corr_matrix = df_train[top_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    ax=ax,
    square=True
)
ax.set_title("Matrice de corrélation - Top 15 features + TARGET")
plt.tight_layout()
plt.show()

In [ ]:
# boxplots - Distribution des features clés par TARGET

features_to_plot = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
                    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'AMT_CREDIT',
                    'AMT_INCOME_TOTAL', 'AMT_ANNUITY']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feature in enumerate(features_to_plot):
    df_train.boxplot(
        column=feature,
        by='TARGET',
        ax=axes[i],
        boxprops=dict(color='steelblue'),
        medianprops=dict(color='red')
    )
    axes[i].set_title(feature, fontsize=10)
    axes[i].set_xlabel("TARGET (0=bon client, 1=défaut)")

plt.suptitle("Distribution des features clés par TARGET", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
total_missing = (missing_df['nb_manquants'] > 0).sum()
print(f"Colonnes avec au moins 1 valeur manquante : {total_missing} / {df_train.shape[1]}")

# répartition par seuil
seuils = [30, 50, 70]
for s in seuils:
    n = (missing_df['pct_manquants'] > s).sum()
    print(f"Colonnes avec plus de {s}% manquants    : {n}")

In [ ]:
print("\n=== Variables catégorielles ===")
cat_cols = df_train.select_dtypes(include='object').columns
for col in cat_cols:
    n_unique = df_train[col].nunique()
    print(f"{col:<40} {n_unique} modalités")

# Suite de l'exploration - tables secondaires

In [ ]:
bureau = pd.read_csv('../data/bureau.csv')
prev_app = pd.read_csv('../data/previous_application.csv')
pos_cash = pd.read_csv('../data/POS_CASH_balance.csv')
installments = pd.read_csv('../data/installments_payments.csv')
credit_card = pd.read_csv('../data/credit_card_balance.csv')

print(f"bureau             : {bureau.shape}")
print(f"previous_application: {prev_app.shape}")
print(f"POS_CASH_balance   : {pos_cash.shape}")
print(f"installments       : {installments.shape}")
print(f"credit_card        : {credit_card.shape}")

In [ ]:
print(bureau.head(3))
print(f"\nColonnes : {bureau.columns.tolist()}")
print(f"\nValeurs manquantes :\n{bureau.isnull().sum()[bureau.isnull().sum() > 0]}")

In [ ]:
# agrégation de bureau par client (SK_ID_CURR)

# -- Partie 1 : variables numériques --
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    # nombre total de crédits passés
    BUREAU_NB_CREDITS        = ('SK_ID_BUREAU', 'count'),
    # nombre de jours depuis le crédit le plus récent
    BUREAU_DAYS_CREDIT_MEAN  = ('DAYS_CREDIT', 'mean'),
    BUREAU_DAYS_CREDIT_MIN   = ('DAYS_CREDIT', 'min'),
    # retards de paiement
    BUREAU_OVERDUE_MEAN      = ('CREDIT_DAY_OVERDUE', 'mean'),
    BUREAU_OVERDUE_MAX       = ('CREDIT_DAY_OVERDUE', 'max'),
    # montants
    BUREAU_AMT_CREDIT_MEAN   = ('AMT_CREDIT_SUM', 'mean'),
    BUREAU_AMT_CREDIT_SUM    = ('AMT_CREDIT_SUM', 'sum'),
    BUREAU_AMT_DEBT_MEAN     = ('AMT_CREDIT_SUM_DEBT', 'mean'),
    BUREAU_AMT_DEBT_SUM      = ('AMT_CREDIT_SUM_DEBT', 'sum'),
).reset_index()

# -- Partie 2 : variables catégorielles --
credit_active_dummies = pd.get_dummies(bureau['CREDIT_ACTIVE'], prefix='BUREAU')
bureau_cat = pd.concat([bureau[['SK_ID_CURR']], credit_active_dummies], axis=1)
bureau_cat_agg = bureau_cat.groupby('SK_ID_CURR').sum().reset_index()

bureau_agg = bureau_agg.merge(bureau_cat_agg, on='SK_ID_CURR', how='left')

print(f"Shape bureau agrégé : {bureau_agg.shape}")
print(bureau_agg.head(3))

In [ ]:
# agrégation - previous_application
prev_agg = prev_app.groupby('SK_ID_CURR').agg(
    PREV_NB_APPLICATIONS     = ('SK_ID_PREV', 'count'),
    PREV_AMT_CREDIT_MEAN     = ('AMT_CREDIT', 'mean'),
    PREV_AMT_CREDIT_SUM      = ('AMT_CREDIT', 'sum'),
    PREV_AMT_ANNUITY_MEAN    = ('AMT_ANNUITY', 'mean'),
    PREV_DAYS_DECISION_MEAN  = ('DAYS_DECISION', 'mean'),
    PREV_DAYS_DECISION_MIN   = ('DAYS_DECISION', 'min'),
).reset_index()

prev_status = pd.get_dummies(prev_app['NAME_CONTRACT_STATUS'], prefix='PREV')
prev_status_agg = pd.concat([prev_app[['SK_ID_CURR']], prev_status], axis=1)
prev_status_agg = prev_status_agg.groupby('SK_ID_CURR').sum().reset_index()

prev_agg = prev_agg.merge(prev_status_agg, on='SK_ID_CURR', how='left')
print(f"Shape previous_application agrégé : {prev_agg.shape}")

In [ ]:
# agrégation - POS_CASH_balance
pos_agg = pos_cash.groupby('SK_ID_CURR').agg(
    POS_NB_MONTHS            = ('MONTHS_BALANCE', 'count'),
    POS_MONTHS_BALANCE_MEAN  = ('MONTHS_BALANCE', 'mean'),
    POS_SK_DPD_MEAN          = ('SK_DPD', 'mean'),      
    POS_SK_DPD_MAX           = ('SK_DPD', 'max'),        
    POS_SK_DPD_DEF_MEAN      = ('SK_DPD_DEF', 'mean'),   
).reset_index()
print(f"Shape POS_CASH agrégé : {pos_agg.shape}")

In [ ]:
# agrégation - installments_payments
installments['PAYMENT_DIFF'] = installments['AMT_INSTALMENT'] - installments['AMT_PAYMENT']
installments['DAYS_DIFF'] = installments['DAYS_INSTALMENT'] - installments['DAYS_ENTRY_PAYMENT']

inst_agg = installments.groupby('SK_ID_CURR').agg(
    INST_NB_PAYMENTS         = ('SK_ID_PREV', 'count'),
    INST_PAYMENT_DIFF_MEAN   = ('PAYMENT_DIFF', 'mean'), 
    INST_PAYMENT_DIFF_MAX    = ('PAYMENT_DIFF', 'max'),
    INST_DAYS_DIFF_MEAN      = ('DAYS_DIFF', 'mean'),
    INST_DAYS_DIFF_MAX       = ('DAYS_DIFF', 'max'),
    INST_AMT_PAYMENT_SUM     = ('AMT_PAYMENT', 'sum'),
).reset_index()
print(f"Shape installments agrégé : {inst_agg.shape}")

In [ ]:
# agrégation - credit_card_balance
cc_agg = credit_card.groupby('SK_ID_CURR').agg(
    CC_NB_MONTHS             = ('MONTHS_BALANCE', 'count'),
    CC_AMT_BALANCE_MEAN      = ('AMT_BALANCE', 'mean'),
    CC_AMT_BALANCE_MAX       = ('AMT_BALANCE', 'max'),
    CC_AMT_DRAWINGS_MEAN     = ('AMT_DRAWINGS_CURRENT', 'mean'),
    CC_SK_DPD_MEAN           = ('SK_DPD', 'mean'),
    CC_SK_DPD_MAX            = ('SK_DPD', 'max'),
).reset_index()
print(f"Shape credit_card agrégé : {cc_agg.shape}")

In [ ]:
print(credit_card.columns.tolist())

In [ ]:
# fusion de toutes les tables avec application_train
df = df_train.copy() 

df = df.merge(bureau_agg,  on='SK_ID_CURR', how='left')
df = df.merge(prev_agg,    on='SK_ID_CURR', how='left')
df = df.merge(pos_agg,     on='SK_ID_CURR', how='left')
df = df.merge(inst_agg,    on='SK_ID_CURR', how='left')
df = df.merge(cc_agg,      on='SK_ID_CURR', how='left')

print(f"Shape final : {df.shape}")
print(f"Shape application_train original : {df_train.shape}")
print(f"Nouvelles features ajoutées : {df.shape[1] - df_train.shape[1]}")

# Prétraitement

In [ ]:
missing_pct = (df.isnull().sum() / len(df)) * 100

cols_to_drop = missing_pct[missing_pct > 60].index.tolist()
print(f"Colonnes supprimées (>60% manquants) : {len(cols_to_drop)}")
print(cols_to_drop)

df = df.drop(columns=cols_to_drop)
print(f"\nShape après suppression : {df.shape}")

In [ ]:
from sklearn.impute import SimpleImputer

# séparer les colonnes numériques et catégorielles
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

# on retire la variable cible des colonnes numériques
num_cols = [c for c in num_cols if c != 'TARGET']

print(f"Colonnes numériques : {len(num_cols)}")
print(f"Colonnes catégorielles : {len(cat_cols)}")

# imputation par la médiane pour les numériques
imp_median = SimpleImputer(strategy='median')
df[num_cols] = imp_median.fit_transform(df[num_cols])

# imputation par la valeur la plus fréquente pour les catégorielles
imp_mode = SimpleImputer(strategy='most_frequent')
df[cat_cols] = imp_mode.fit_transform(df[cat_cols])

# vérification
remaining_missing = df.isnull().sum().sum()
print(f"Valeurs manquantes restantes : {remaining_missing}")

In [ ]:
# encodage des variables catégorielles
from sklearn.preprocessing import LabelEncoder

# pour les colonnes avec seulement 2 modalités : LabelEncoder 
le = LabelEncoder()
le_cols = []
ohe_cols = []

for col in cat_cols:
    n_unique = df[col].nunique()
    if n_unique <= 2:
        df[col] = le.fit_transform(df[col])
        le_cols.append(col)
    else:
        ohe_cols.append(col)

print(f"Colonnes encodées avec LabelEncoder : {le_cols}")
print(f"Colonnes à encoder avec OneHotEncoder : {ohe_cols}")

# OneHotEncoding pour les colonnes avec plus de 2 modalités
df = pd.get_dummies(df, columns=ohe_cols)

print(f"\nShape après encodage : {df.shape}")

In [ ]:
# normalisation des variables numériques
from sklearn.preprocessing import StandardScaler

cols_to_scale = [c for c in df.select_dtypes(include=['float64', 'int64']).columns
                 if c != 'TARGET'
                 and c != 'SK_ID_CURR'
                 and df[c].nunique() > 2] 

print(f"Colonnes à normaliser : {len(cols_to_scale)}")

scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print(f"Aperçu après normalisation :")
df[cols_to_scale[:3]].describe().round(2)

In [ ]:
# nettoyage des noms de colonnes (compatibilité LightGBM)
import re

df.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in df.columns]
print(df.columns.tolist()[:10])

In [ ]:
# sélection de features - Étape 1 : quasi-constantes
from sklearn.feature_selection import VarianceThreshold

X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

# seuil très bas : on élimine uniquement les colonnes vraiment figées
selector = VarianceThreshold(threshold=0.01)
selector.fit(X)

cols_kept = X.columns[selector.get_support()].tolist()
cols_dropped = X.columns[~selector.get_support()].tolist()

print(f"Features avant : {X.shape[1]}")
print(f"Supprimées (quasi-constantes) : {len(cols_dropped)}")
print(f"Features restantes : {len(cols_kept)}")

X = X[cols_kept]

In [ ]:
# sélection de features - Étape 2 : corrélations inter-features

corr_matrix = X.corr().abs()

# matrice triangulaire supérieure
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# colonnes avec corrélation > 0.95 avec une autre
cols_to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]

print(f"Supprimées (corrélation > 0.95) : {len(cols_to_drop_corr)}")

X = X.drop(columns=cols_to_drop_corr)
print(f"Features restantes : {X.shape[1]}")

In [ ]:
# sélection de features - Étape 3 : importance LightGBM
import lightgbm as lgb
from sklearn.model_selection import train_test_split

X_train_sel, X_test_sel, y_train_sel, y_test_sel = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

lgbm_selector = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm_selector.fit(X_train_sel, y_train_sel)

feature_importance_sel = pd.DataFrame({
    'feature'   : X.columns,
    'importance': lgbm_selector.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 30 features :")
print(feature_importance_sel.head(30).to_string(index=False))

In [ ]:
# sélection de features - Étape 4 : top 25
TOP_N = 25
top_features = feature_importance_sel.head(TOP_N)['feature'].tolist()

print(f"Features sélectionnées ({TOP_N}) :")
for i, f in enumerate(top_features, 1):
    print(f"  {i:2}. {f}")

# reconstruction du df final avec uniquement les features retenues
df = df[['SK_ID_CURR', 'TARGET'] + top_features]
print(f"\nShape final df : {df.shape}")

In [ ]:
df.to_csv('../data/application_train_prepared.csv', index=False)
print(f"Shape final : {df.shape}")